In [ ]:
from odc.stac import load
from pystac.client import Client
from ipyleaflet import basemaps
import folium

from utils import make_indices, mask_land, locations, mask_deeps

In [ ]:
# Reload scripts and imports
%load_ext autoreload
%autoreload 2

In [ ]:
client = Client.open("https://stac.digitalearthpacific.org")
collection = "dep_s2_geomad"

location = locations.malolo

items = client.search(collections=[collection], bbox=location.bbox, datetime="2024").item_collection()

print(f"Found {len(items)} items")

In [ ]:
data = load(items, bbox=location.bbox, bands=["red", "green", "blue", "nir", "swir16"])
data = make_indices(data).squeeze()

data

In [ ]:
# This mask uses a combination of the ln(b/g) and the stumpf index
masked, mask = mask_deeps(data, return_mask=True, stumpf_threshold=1.9)

In [ ]:
double_masked = mask_land(masked)

In [ ]:
centroid = data.geobox.geographic_extent.centroid.coords[0][::-1]

m = folium.Map(location=centroid, zoom_start=12)
_ = folium.TileLayer(tiles=basemaps.Esri.WorldImagery).add_to(m)

data.odc.explore(m, vmin=0, vmax=2000, name="RGB")
masked.odc.explore(m, vmin=0, vmax=2000, name="Deep Water Masked")
double_masked.odc.explore(m, vmin=0, vmax=2000, name="Water and Land Masked")

folium.LayerControl().add_to(m)

m